# Olist E-Commerce — Exploratory Data Analysis
### IT5006 Group Project · shared EDA notebook

**Purpose of this notebook:** explore all 9 Olist tables, surface insights, and give the group a
common *base table* to build on. Bring the charts + the "Insights" section at the bottom to the meeting.

**How to run (Colab):**
1. Put the dataset in your Drive at `MyDrive/IT5006_Project-Data/Olist_CSV/`
2. Run the "Mount Drive" cell and authorise.
3. Runtime ▸ Run all.

Sections: **A** data quality · **B** build the base table · **C** insights & charts · **D** takeaways & next steps.

## Setup

In [ ]:
# --- Mount Google Drive (Colab only) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not in Colab - will read from a local path instead.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

# >>> Data location (edit only if your Drive layout differs) <<<
DATA_DIR = "/content/drive/MyDrive/IT5006_Project-Data/Olist_CSV"
if not IN_COLAB:
    DATA_DIR = "Olist_CSV"   # fallback for running locally

assert os.path.isdir(DATA_DIR), f"Folder not found: {DATA_DIR}"
print("Reading data from:", DATA_DIR)
print("Files:", sorted(os.listdir(DATA_DIR)))

### Load all tables
Date columns are parsed on load so we can do time-based analysis later.

In [ ]:
files = {
    "customers":       "olist_customers_dataset.csv",
    "geolocation":     "olist_geolocation_dataset.csv",
    "orders":          "olist_orders_dataset.csv",
    "order_items":     "olist_order_items_dataset.csv",
    "payments":        "olist_order_payments_dataset.csv",
    "reviews":         "olist_order_reviews_dataset.csv",
    "products":        "olist_products_dataset.csv",
    "sellers":         "olist_sellers_dataset.csv",
    "cat_translation": "product_category_name_translation.csv",
}
date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
df = {k: pd.read_csv(os.path.join(DATA_DIR, fn), parse_dates=date_cols.get(k))
      for k, fn in files.items()}
for k, d in df.items():
    print(f"{k:16s} {d.shape[0]:>8,} rows x {d.shape[1]} cols")

---
## Part A — Data quality profile

Before any modelling we check **shape, missing values, and duplicates** for every table.
The numbers below tell the group what needs cleaning and which columns are unreliable.

In [ ]:
profile = []
for k, d in df.items():
    profile.append({"table": k, "rows": len(d), "cols": d.shape[1],
                    "total_missing": int(d.isna().sum().sum()),
                    "dup_rows": int(d.duplicated().sum())})
pd.DataFrame(profile).set_index("table")

**Which columns are actually missing values, and how badly:**

In [ ]:
rows = []
for k, d in df.items():
    m = d.isna().sum()
    m = m[m > 0]
    for col, n in m.items():
        rows.append({"table": k, "column": col, "n_missing": int(n),
                     "pct": round(n/len(d)*100, 1)})
pd.DataFrame(rows).sort_values("pct", ascending=False).reset_index(drop=True)

> **How to read this — the data-quality story for the meeting:**
> - **Reviews text is mostly empty** (`review_comment_title` ~88%, `review_comment_message` ~59% missing).
>   → Use the numeric **review_score**, not the text, unless someone specifically wants to do NLP.
> - **~3% of orders have no delivery date** (`order_delivered_customer_date`). These are orders that were
>   cancelled / never delivered. → Filter to `order_status == 'delivered'` for any delivery-time work.
> - **Products** have ~1.9% missing category and a couple of rows with missing weight/dimensions. Small — impute or drop.
> - **Geolocation has ~262k exact duplicate rows** — we collapse it to one row per zip code in Part B.

**A few key integrity facts worth mentioning:**

In [ ]:
print("customer_id unique per row :", df['customers']['customer_id'].is_unique)
print("distinct real customers     :", df['customers']['customer_unique_id'].nunique(), "(vs", len(df['customers']), "customer_id rows)")
print()
print("order_status counts:")
print(df['orders']['order_status'].value_counts())
print()
print("distinct orders in order_items:", df['order_items']['order_id'].nunique(),
      "| max items in one order:", df['order_items']['order_item_id'].max())

> **Key gotcha:** `customer_id` is created *per order*, so it looks unique but it is **not the person**.
> The real customer is `customer_unique_id` (96,096 people across 99,441 orders). Use `customer_unique_id`
> for anything about repeat buyers.

---
## Part B — Build the analytical base table

Most interesting questions need several tables joined together. Here we build **one row per order-item**
(the finest useful grain) with everything attached: order info, product physical attributes, customer &
seller location, shipping distance, and freight economics. **This is the shared starting point for the team.**

**Step 1 — Collapse geolocation to one clean row per zip code** (it has 262k duplicates), then attach the
English product category and compute physical size features.

*Volumetric weight* = `L × W × H / 5000` — the industry formula couriers use to bill bulky-but-light parcels.

In [ ]:
# geolocation -> mean lat/lng per zip prefix
geo = (df["geolocation"]
       .groupby("geolocation_zip_code_prefix")
       .agg(geo_lat=("geolocation_lat", "mean"), geo_lng=("geolocation_lng", "mean"))
       .reset_index())

# products + english name + size features
prod = df["products"].merge(df["cat_translation"], on="product_category_name", how="left")
prod["product_category_english"] = (prod["product_category_name_english"]
                                     .fillna(prod["product_category_name"]).fillna("unknown"))
prod["volume_cm3"] = prod["product_length_cm"] * prod["product_height_cm"] * prod["product_width_cm"]
prod["volumetric_weight_kg"] = prod["volume_cm3"] / 5000.0
prod["weight_kg"] = prod["product_weight_g"] / 1000.0
prod[["product_category_english", "volume_cm3", "volumetric_weight_kg", "weight_kg"]].describe()

**Step 2 — Join everything onto `order_items`** (order → product → customer → seller → lat/lng).

In [ ]:
base = (df["order_items"]
        .merge(df["orders"], on="order_id", how="left")
        .merge(prod[["product_id", "product_category_english", "volume_cm3",
                     "volumetric_weight_kg", "weight_kg"]], on="product_id", how="left")
        .merge(df["customers"][["customer_id", "customer_zip_code_prefix",
                                "customer_city", "customer_state"]], on="customer_id", how="left")
        .merge(df["sellers"][["seller_id", "seller_zip_code_prefix",
                              "seller_city", "seller_state"]], on="seller_id", how="left"))

# attach lat/lng for both ends
base = base.merge(geo.rename(columns={"geolocation_zip_code_prefix": "customer_zip_code_prefix",
                                      "geo_lat": "cust_lat", "geo_lng": "cust_lng"}),
                  on="customer_zip_code_prefix", how="left")
base = base.merge(geo.rename(columns={"geolocation_zip_code_prefix": "seller_zip_code_prefix",
                                      "geo_lat": "sell_lat", "geo_lng": "sell_lng"}),
                  on="seller_zip_code_prefix", how="left")
print("base table:", base.shape)

**Step 3 — Engineer the freight-economics features.**
- `distance_km` — great-circle (Haversine) distance from seller to customer.
- `freight_ratio` — freight ÷ price (how much of the item's price the shipping adds).
- `high_freight_flag` — a candidate **classification target**: is freight more than 25% of the price?
- `delivery_days` — order-to-doorstep time (context for the temporal angle).

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

base["distance_km"]  = haversine(base["sell_lat"], base["sell_lng"], base["cust_lat"], base["cust_lng"])
base["freight_ratio"] = base["freight_value"] / base["price"].replace(0, np.nan)

THRESHOLD = 0.25
base["high_freight_flag"] = (base["freight_ratio"] > THRESHOLD).astype("Int64")
base["delivery_days"]  = (base["order_delivered_customer_date"] - base["order_purchase_timestamp"]).dt.total_seconds()/86400
base["purchase_month"] = base["order_purchase_timestamp"].dt.to_period("M").astype(str)

print("Feature coverage (non-null %):")
for c in ["freight_value","price","freight_ratio","volumetric_weight_kg","weight_kg","distance_km","product_category_english"]:
    print(f"  {c:26s} {base[c].notna().mean()*100:5.1f}%")
base.head(3)

> **Optional — save the base table to Drive** so teammates can load it directly and skip the joins.
> Uncomment to run.

In [ ]:
# out = "/content/drive/MyDrive/IT5006_Project-Data/order_item_base_table.csv"
# base.to_csv(out, index=False)
# print("saved ->", out)

---
## Part C — Insights & charts

Each chart has a short note on **what it shows** and **why it matters** for the project.

In [ ]:
# clean working subset for the freight charts
f = base.dropna(subset=["freight_ratio"])
f = f[(f["price"] > 0) & (f["freight_value"] >= 0)].copy()
print("rows used for freight charts:", len(f))

### C1 · Order status — is the dataset mostly completed orders?

In [ ]:
plt.figure(figsize=(8,4))
vc = df["orders"]["order_status"].value_counts()
sns.barplot(x=vc.values, y=vc.index, color="#4c72b0")
plt.xscale("log"); plt.xlabel("count (log scale)"); plt.ylabel("")
plt.title("Order status distribution"); plt.show()

**Reading it:** ~97% of orders are `delivered`; the rest (shipped, canceled, unavailable…) are a small tail.
**So what:** the dataset is clean enough to model on, but if you study delivery time you must drop the
non-delivered orders (they have no delivery date).

### C2 · Price and freight are heavily right-skewed

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(np.log1p(f["price"]), bins=60, ax=ax[0], color="#55a868")
ax[0].set_title("log(1 + price)"); ax[0].set_xlabel("log price")
sns.histplot(np.log1p(f["freight_value"]), bins=60, ax=ax[1], color="#dd8452")
ax[1].set_title("log(1 + freight_value)"); ax[1].set_xlabel("log freight")
plt.show()
print("price   -> median R$%.2f | mean R$%.2f | p95 R$%.2f" % (f['price'].median(), f['price'].mean(), f['price'].quantile(.95)))
print("freight -> median R$%.2f | mean R$%.2f" % (f['freight_value'].median(), f['freight_value'].mean()))

**Reading it:** both distributions have a long right tail (a few very expensive items). We plot them on a
**log scale** so the shape is visible. **So what:** for regression, model `log(price)` / `log(freight)`,
not the raw values — otherwise a handful of outliers dominate the fit.

### C3 · Freight-to-price ratio — the freight burden

In [ ]:
plt.figure(figsize=(9,4))
sns.histplot(f["freight_ratio"].clip(upper=2), bins=80, color="#8172b3")
plt.axvline(THRESHOLD, color="red", ls="--", label=f"{int(THRESHOLD*100)}% threshold")
plt.xlabel("freight / price   (clipped at 2.0)"); plt.legend()
plt.title("Freight-to-price ratio"); plt.show()
print("median ratio: %.3f | mean ratio: %.3f" % (f['freight_ratio'].median(), f['freight_ratio'].mean()))
print("share above %d%%: %.1f%%" % (THRESHOLD*100, f['high_freight_flag'].mean()*100))

**Reading it:** the typical order pays **~23% of the item's price again in shipping**; ~46% of items are
above the 25% line. **So what:** shipping cost is a genuine business pain-point — a strong, non-generic project
angle. Note ~46% positive means the classification target is close to balanced (no heavy imbalance to fix).

### C4 · What actually drives freight? (weight & volume)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
samp = f.sample(min(20000, len(f)), random_state=1)
ax[0].scatter(samp["weight_kg"], samp["freight_value"], s=5, alpha=0.2, color="#4c72b0")
ax[0].set_xlim(0, samp["weight_kg"].quantile(.99)); ax[0].set_ylim(0, samp["freight_value"].quantile(.99))
ax[0].set_xlabel("weight (kg)"); ax[0].set_ylabel("freight (R$)"); ax[0].set_title("Freight vs actual weight")
ax[1].scatter(samp["volumetric_weight_kg"], samp["freight_value"], s=5, alpha=0.2, color="#dd8452")
ax[1].set_xlim(0, samp["volumetric_weight_kg"].quantile(.99)); ax[1].set_ylim(0, samp["freight_value"].quantile(.99))
ax[1].set_xlabel("volumetric weight (kg)"); ax[1].set_ylabel("freight (R$)"); ax[1].set_title("Freight vs volumetric weight")
plt.show()

**Reading it:** freight rises clearly with both real weight and volumetric weight — the upward cloud.
**So what:** this is the evidence that a **fair-freight regression model** is feasible: shipping cost is a
function of the physical parcel, which we can compute from the product table.

### C5 · Freight vs shipping distance

In [ ]:
plt.figure(figsize=(8,5))
s2 = f.dropna(subset=["distance_km"]).sample(min(20000, f["distance_km"].notna().sum()), random_state=1)
plt.scatter(s2["distance_km"], s2["freight_value"], s=5, alpha=0.2, color="#55a868")
plt.ylim(0, s2["freight_value"].quantile(.99))
plt.xlabel("seller -> customer distance (km)"); plt.ylabel("freight (R$)")
plt.title("Freight vs shipping distance (Haversine)"); plt.show()

**Reading it:** longer distances trend toward higher freight, but with lots of spread (Brazil is huge;
some sellers ship nationwide). **So what:** distance is a useful predictor but **weight/volume matter more**
(see the heatmap in C7). Distance is where the *geolocation* table earns its place.

### C6 · Which product categories carry the highest freight burden?

In [ ]:
plt.figure(figsize=(9,6))
catr = (f.groupby("product_category_english")
          .agg(median_ratio=("freight_ratio","median"), n=("freight_ratio","size")))
catr = catr[catr["n"] >= 200].sort_values("median_ratio", ascending=False).head(15)
sns.barplot(x=catr["median_ratio"], y=catr.index, color="#c44e52")
plt.xlabel("median freight / price ratio"); plt.ylabel("")
plt.title("Top 15 categories by freight burden (n >= 200)"); plt.show()

**Reading it:** electronics, telephony and food/drink pay the most freight *relative to price* — these are
cheap-but-bulky/heavy goods. **So what:** product **category** is a strong categorical feature for both the
classifier and the regressor.

### C7 · Correlation of the numeric drivers

In [ ]:
plt.figure(figsize=(7,6))
num = f[["price","freight_value","freight_ratio","weight_kg","volumetric_weight_kg","distance_km","delivery_days"]]
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation — freight economics drivers"); plt.show()

**Reading it (the single most useful chart):**
- `freight_value` correlates **0.61 with weight**, **0.59 with volumetric weight**, **0.39 with distance**,
  but only **0.41 with price** → freight is about the *parcel*, not the item's price.
- `weight_kg` and `volumetric_weight_kg` correlate **0.80** with each other → **multicollinearity**; don't put
  both raw into a linear model (combine them, or use a tree model).
**So what:** this justifies the feature choices for the regression model in one picture.

### C8 · Order volume over time (seasonality)

In [ ]:
plt.figure(figsize=(11,4))
monthly = (df["orders"].assign(m=df["orders"]["order_purchase_timestamp"].dt.to_period("M"))
           .groupby("m").size())
monthly.index = monthly.index.astype(str)
sns.lineplot(x=monthly.index, y=monthly.values, marker="o", color="#4c72b0")
plt.xticks(rotation=90); plt.ylabel("orders"); plt.xlabel("")
plt.title("Order volume by month"); plt.show()

**Reading it:** the business grows through 2017 and peaks around **Nov 2017 (Black Friday)**; the series
mostly covers **late 2016 → mid/late 2018**. **So what:** there is real seasonality — a time feature (month,
is-holiday) is worth adding, and any train/test split should respect time order.

### C9 · Payment methods & review scores (context)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
pv = df["payments"]["payment_type"].value_counts()
sns.barplot(x=pv.values, y=pv.index, ax=ax[0], color="#8172b3")
ax[0].set_title("Payment type"); ax[0].set_xlabel("count"); ax[0].set_ylabel("")
rv = df["reviews"]["review_score"].value_counts().sort_index()
sns.barplot(x=rv.index, y=rv.values, ax=ax[1], color="#ccb974")
ax[1].set_title("Review score"); ax[1].set_xlabel("stars"); ax[1].set_ylabel("count")
plt.show()

**Reading it:** credit card dominates payments; reviews are **bimodal** — mostly 5-star with a spike of
1-star. **So what:** if the group prefers a *satisfaction* angle instead of freight, "what drives a 1-star
review?" is a viable alternative classification problem (delivery delay is a known driver — worth testing).

---
## Part D — Takeaways & what to decide as a group

### Insights to present
1. **Shipping is expensive relative to price** — median freight is ~23% of item price; ~46% of items exceed 25%.
2. **Freight is driven by weight / volume / distance, not price** (corr 0.61 / 0.59 / 0.39 vs 0.41) — so a
   *fair-freight* prediction model is well-motivated.
3. **Category matters** — electronics, telephony, food/drink carry the heaviest freight burden.
4. **Data-quality flags** — review text ~88% empty, ~3% of orders undelivered, geolocation had 262k dup rows.

### Two candidate problem statements (bring both, let the group pick)
| | Problem | Classification | Regression | Risk |
|---|---|---|---|---|
| **A** | **Freight economics** | Is this a high-freight-ratio order? | Predict a fair freight value | distinctive, medium effort |
| **B** | **Delivery / satisfaction** | Will the order be late / 1-star? | Predict delivery days | common & safe |

### Open decisions for the meeting
- **Which problem?** (A is more original; B is safer.)
- **How to split work?** Suggestion: split by **feature domain on this shared base table** (spatial /
  temporal / product / financial / seller-review) rather than by raw CSV — otherwise people block on each other.
- **Setup:** agree on Colab + Drive path + a GitHub repo so everyone runs the same code.

---
## Appendix — per-person work areas (fill in after the group agrees)
*Lightweight stubs so each person has a place to start once the split is decided. Not full analyses yet.*

### Person 1 — Spatial & geographic
_Ideas:_ clean coordinates, map trade routes, refine distance_km, regional freight differences

In [ ]:
# Person 1: your exploration here (uses `base`, `df[...]`)


### Person 2 — Temporal & logistics
_Ideas:_ lead times (approval/dispatch/transit), seasonality, late-delivery rate

In [ ]:
# Person 2: your exploration here (uses `base`, `df[...]`)


### Person 3 — Financial & freight
_Ideas:_ price vs freight ratio, payments, installments, basket size, log transforms

In [ ]:
# Person 3: your exploration here (uses `base`, `df[...]`)


### Person 4 — Product & physical
_Ideas:_ dimension cleaning, volumetric weight, category clustering

In [ ]:
# Person 4: your exploration here (uses `base`, `df[...]`)


### Person 5 — Seller & reviews
_Ideas:_ seller volume (Pareto), review drivers, seller response time

In [ ]:
# Person 5: your exploration here (uses `base`, `df[...]`)
